In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Impact of Error Types on Decoder Performance
# ## Isolated Error Analysis for Composite DNA Decoding
# ## All Error Models: EZ17, G15, O17
# ## Alphabet: A_11 (15 classes), Coverage: M=10

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "3"  # Change as needed
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

# Fixed parameters for this analysis
ALPHABET_MODE = "2mix_3mix_4mix"  # A_11 alphabet
COVERAGE_M = 10

# All error models to analyze
ERROR_MODELS_TO_ANALYZE = ["erlich", "grass", "organick"]

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Vocabulary sizes
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

# Base configuration (error model will be set per experiment)
BASE_CONFIG = {
    # Alphabet Mode
    "alphabet_mode": ALPHABET_MODE,
    
    # Vocabulary
    "vocab_size": VOCAB_SIZES[ALPHABET_MODE],
    
    # Coverage for this analysis
    "coverage_M": COVERAGE_M,
    
    # Dataset Parameters for isolated error analysis
    "num_samples_eval": 20000,  # Number of samples for evaluation
    
    # Model Architecture (must match trained model)
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Batch size for evaluation
    "batch_size": 500,
    
    # Reproducibility
    "seed": 42
}

# Output directory
OUTPUT_DIR = f"./isolated_error_analysis_{ALPHABET_MODE}_M{COVERAGE_M}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"{'='*60}")
print(f"📋 ISOLATED ERROR TYPE ANALYSIS CONFIGURATION")
print(f"{'='*60}")
print(f"   Alphabet: {ALPHABET_MODE} ({BASE_CONFIG['vocab_size']} classes)")
print(f"   Coverage Depth: {COVERAGE_M}")
print(f"   Error Models: {[ERROR_MODEL_SPECS[m]['name'] for m in ERROR_MODELS_TO_ANALYZE]}")
print(f"   Evaluation Samples: {BASE_CONFIG['num_samples_eval']:,}")
print(f"   Output Dir: {OUTPUT_DIR}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(BASE_CONFIG['seed'])
print(f"🎲 Random seed set to: {BASE_CONFIG['seed']}")


# =============================================================================
# CELL 5: SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================

def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    symbol_to_idx.update({'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9})
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({'Q1': 14})
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
        [0.5, 0.0, 0.0, 0.5], [0.0, 0.5, 0.5, 0.0],
        [0.0, 0.5, 0.0, 0.5], [0.0, 0.0, 0.5, 0.5],
        [0.5, 0.5, 0.0, 0.0], [0.5, 0.0, 0.5, 0.0],
    ]
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0], [third, third, 0.0, third],
            [third, 0.0, third, third], [0.0, third, third, third],
        ])
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_symbol_to_idx(BASE_CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(BASE_CONFIG["alphabet_mode"]).to(device)

print(f"\n📊 Symbol Mappings: {len(SYMBOL_TO_IDX)} symbols")


# =============================================================================
# CELL 6: COMPOSITE ALPHABET DEFINITIONS
# =============================================================================

PURE_BASES = {'A': ['A'], 'C': ['C'], 'G': ['G'], 'T': ['T']}

TWO_MIX_MAP = {
    'M1': ['A', 'T'], 'M2': ['C', 'G'], 'M3': ['C', 'T'],
    'M4': ['G', 'T'], 'M5': ['A', 'C'], 'M6': ['A', 'G'],
}

THREE_MIX_MAP = {
    'T1': ['A', 'C', 'G'], 'T2': ['A', 'C', 'T'],
    'T3': ['A', 'G', 'T'], 'T4': ['C', 'G', 'T'],
}

FOUR_MIX_MAP = {'Q1': ['A', 'C', 'G', 'T']}


def build_composite_map(mode):
    """Build the composite symbol mapping based on alphabet mode."""
    composite_map = PURE_BASES.copy()
    composite_map.update(TWO_MIX_MAP)
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        composite_map.update(THREE_MIX_MAP)
    if mode == "2mix_3mix_4mix":
        composite_map.update(FOUR_MIX_MAP)
    return composite_map


COMPOSITE_MAP = build_composite_map(BASE_CONFIG["alphabet_mode"])
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())

print(f"🧬 Composite Alphabet: {len(ALL_SYMBOLS)} symbols")


# =============================================================================
# CELL 7: ERROR RATES CLASS WITH ISOLATED ERROR SUPPORT (ALL 3 MODELS)
# =============================================================================

class ErrorRatesIsolated:
    """
    Error rate configuration with support for isolated error types.
    Supports EZ17, G15, and O17 error profiles.
    
    Modes:
        - "all": All error types active (original)
        - "substitution_only": Only substitution errors
        - "insertion_only": Only insertion errors  
        - "deletion_only": Only deletion errors
    """
    
    def __init__(self, error_mode="all"):
        """
        Initialize error rates.
        
        Args:
            error_mode: One of "all", "substitution_only", "insertion_only", "deletion_only"
        """
        self.error_mode = error_mode
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0}
        }
    
    def _apply_error_mode(self, base_general, base_per_base):
        """Apply error mode filtering to base error rates."""
        
        if self.error_mode == "all":
            # All errors active
            self.general_errors = base_general.copy()
            self.per_base_errors = {k: v.copy() for k, v in base_per_base.items()}
            
        elif self.error_mode == "substitution_only":
            # Only substitution errors (p_i = p_d = 0)
            self.general_errors = {
                's': base_general['s'],
                'i': 0.0,
                'd': 0.0,
                'ld': 0.0
            }
            for base in ['A', 'C', 'G', 'T']:
                self.per_base_errors[base] = {
                    's': base_per_base[base]['s'],
                    'i': 0.0,
                    'pi': 0.0,
                    'd': 0.0,
                    'ld': 0.0
                }
                
        elif self.error_mode == "insertion_only":
            # Only insertion errors (p_s = p_d = 0)
            self.general_errors = {
                's': 0.0,
                'i': base_general['i'],
                'd': 0.0,
                'ld': 0.0
            }
            for base in ['A', 'C', 'G', 'T']:
                self.per_base_errors[base] = {
                    's': 0.0,
                    'i': base_per_base[base]['i'],
                    'pi': base_per_base[base].get('pi', base_per_base[base]['i']),
                    'd': 0.0,
                    'ld': 0.0
                }
                
        elif self.error_mode == "deletion_only":
            # Only deletion errors (p_s = p_i = 0)
            self.general_errors = {
                's': 0.0,
                'i': 0.0,
                'd': base_general['d'],
                'ld': base_general['ld']
            }
            for base in ['A', 'C', 'G', 'T']:
                self.per_base_errors[base] = {
                    's': 0.0,
                    'i': 0.0,
                    'pi': 0.0,
                    'd': base_per_base[base]['d'],
                    'ld': base_per_base[base]['ld']
                }
        else:
            raise ValueError(f"Unknown error_mode: {self.error_mode}")
    
    def set_EZ17_values(self):
        """Set Erlich (EZ17) error profile - Illumina MiSeq + Twist Bioscience."""
        
        base_general = {
            's': 1.32e-03,
            'i': 5.81e-04,
            'd': 9.58e-04,
            'ld': 2.33e-04
        }
        
        base_per_base = {
            'A': {'s': 0.00135, 'i': 0.00057, 'pi': 0.00059, 'd': 0.00099, 'ld': 0.00024},
            'C': {'s': 0.00135, 'i': 0.00059, 'pi': 0.00058, 'd': 0.00098, 'ld': 0.00023},
            'G': {'s': 0.00126, 'i': 0.00059, 'pi': 0.00057, 'd': 0.00094, 'ld': 0.00023},
            'T': {'s': 0.00132, 'i': 0.00058, 'pi': 0.00058, 'd': 0.00096, 'ld': 0.00023}
        }
        
        self._apply_error_mode(base_general, base_per_base)
    
    def set_G15_values(self):
        """Set Grass (G15) error profile - Illumina MiSeq + CustomArray."""
        
        base_general = {
            's': 5.84e-03,
            'i': 8.57e-04,
            'd': 5.37e-03,
            'ld': 3.48e-04
        }
        
        base_per_base = {
            'A': {'s': 0.00605, 'i': 0.0009, 'pi': 0.00092, 'd': 0.00543, 'ld': 0.00036},
            'C': {'s': 0.00563, 'i': 0.00083, 'pi': 0.00081, 'd': 0.00513, 'ld': 0.00034},
            'G': {'s': 0.00577, 'i': 0.00085, 'pi': 0.00087, 'd': 0.00539, 'ld': 0.00034},
            'T': {'s': 0.00591, 'i': 0.00084, 'pi': 0.00084, 'd': 0.00559, 'ld': 0.00036}
        }
        
        self._apply_error_mode(base_general, base_per_base)
    
    def set_O17_values(self):
        """Set Organick (O17) error profile - Illumina NextSeq + Twist Bioscience."""
        
        base_general = {
            's': 2.52e-03,
            'i': 4.14e-04,
            'd': 6.94e-04,
            'ld': 2.11e-04
        }
        
        base_per_base = {
            'A': {'s': 0.00717, 'i': 0.0003, 'pi': 0.0012, 'd': 0.00201, 'ld': 0.00054},
            'C': {'s': 0.00034, 'i': 0.00007, 'pi': 0.00007, 'd': 0.00006, 'ld': 0.00001},
            'G': {'s': 0.00196, 'i': 0.00125, 'pi': 0.00029, 'd': 0.00058, 'ld': 0.00023},
            'T': {'s': 0.00055, 'i': 0.00006, 'pi': 0.00009, 'd': 0.00014, 'ld': 0.00006}
        }
        
        self._apply_error_mode(base_general, base_per_base)
    
    def set_values_by_model(self, model_name):
        """Set error values based on model name."""
        if model_name == "erlich":
            self.set_EZ17_values()
        elif model_name == "grass":
            self.set_G15_values()
        elif model_name == "organick":
            self.set_O17_values()
        else:
            raise ValueError(f"Unknown error model: {model_name}")
    
    def print_current_values(self):
        print(f"\n   --- Error Configuration ({self.error_mode}) ---")
        print(f"   General: s={self.general_errors['s']:.2e}, "
              f"i={self.general_errors['i']:.2e}, "
              f"d={self.general_errors['d']:.2e}, "
              f"ld={self.general_errors['ld']:.2e}")
        for base in ['A', 'C', 'G', 'T']:
            rates = self.per_base_errors[base]
            print(f"   {base}: sub={rates['s']:.5f}, ins={rates['i']:.5f}, del={rates['d']:.5f}")
        print("   " + "-"*50)


# =============================================================================
# CELL 8: SEQUENCE GENERATION FUNCTIONS
# =============================================================================

def generate_composite_sequence(length, composite_map):
    """Generates a random sequence of composite symbols."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]


def realize_sequence(composite_seq, composite_map):
    """
    Converts composite symbols to a single DNA realization.
    Each composite symbol is realized by uniformly picking one of its nucleotides.
    """
    realized = []
    for sym in composite_seq:
        nucleotide = random.choice(composite_map[sym])
        realized.append(nucleotide)
    return "".join(realized)


def apply_ids_noise(sequence, error_profile):
    """
    Apply Insertion, Deletion, Substitution noise to a DNA sequence.
    
    Args:
        sequence: Clean DNA string
        error_profile: ErrorRatesIsolated object with per-base error rates
    
    Returns:
        Noisy DNA string
    """
    bases = ['A', 'C', 'G', 'T']
    noisy_seq = []
    
    for base in sequence:
        if base not in bases:
            continue
        
        rates = error_profile.per_base_errors[base]
        p_sub = rates['s']
        p_ins = rates['i']
        p_del = rates['d']
        
        # 1. DELETION Check
        if random.random() < p_del:
            continue  # Skip this base (deletion)
            
        # 2. INSERTION Check (pre-insertion)
        if random.random() < p_ins:
            noisy_seq.append(random.choice(bases))
            
        # 3. SUBSTITUTION vs MATCH Check
        if random.random() < p_sub:
            options = [b for b in bases if b != base]
            noisy_seq.append(random.choice(options))
        else:
            noisy_seq.append(base)
            
    return "".join(noisy_seq)


# =============================================================================
# CELL 9: DATASET GENERATION FOR ISOLATED ERRORS
# =============================================================================

def generate_isolated_error_dataset(config, composite_map, symbol_to_idx, 
                                     error_model, error_mode, verbose=True):
    """
    Generate dataset with isolated error type.
    
    Args:
        config: Configuration dictionary
        composite_map: Composite alphabet mapping
        symbol_to_idx: Symbol to index mapping
        error_model: One of "erlich", "grass", "organick"
        error_mode: One of "all", "substitution_only", "insertion_only", "deletion_only"
        verbose: Print progress
    
    Returns:
        Dictionary with 'metadata' and 'data' keys
    """
    # Setup error profile with isolated errors
    errors = ErrorRatesIsolated(error_mode=error_mode)
    errors.set_values_by_model(error_model)
    
    if verbose:
        errors.print_current_values()
    
    num_samples = config['num_samples_eval']
    seq_length = config['seq_length']
    coverage = config['coverage_M']
    
    dataset = {
        'metadata': {
            'type': f'Composite DNA (Isolated Error: {error_mode})',
            'error_profile': f'{config["error_name"]} - {error_mode}',
            'error_model': error_model,
            'error_mode': error_mode,
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'alphabet_mode': config['alphabet_mode'],
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'seed': config['seed']
        },
        'data': []
    }
    
    if verbose:
        print(f"\n   Generating {num_samples:,} samples...")
    
    start_time = time.time()
    
    for i in range(num_samples):
        # A. Generate Ground Truth (Label) - composite sequence
        clean_composite_seq = generate_composite_sequence(seq_length, composite_map)
        
        # B. Generate Cluster (Input) - multiple noisy reads
        cluster_reads = []
        for _ in range(coverage):
            # 1. Realize: Composite -> DNA
            realized_dna = realize_sequence(clean_composite_seq, composite_map)
            # 2. Corrupt: DNA -> Noisy DNA
            noisy_read = apply_ids_noise(realized_dna, errors)
            cluster_reads.append(noisy_read)
            
        # C. Store sample
        sample = {
            'id': i,
            'label': clean_composite_seq,
            'cluster': cluster_reads
        }
        dataset['data'].append(sample)
        
        # Progress logging
        if verbose and (i + 1) % 5000 == 0:
            elapsed = time.time() - start_time
            samples_per_sec = (i + 1) / elapsed
            eta = (num_samples - i - 1) / samples_per_sec
            print(f"      Processed {i+1:,}/{num_samples:,} | "
                  f"Speed: {samples_per_sec:.1f} samples/s | "
                  f"ETA: {eta:.1f}s")

    total_time = time.time() - start_time
    
    if verbose:
        print(f"      ✅ Complete: {total_time:.1f}s")
    
    return dataset


# =============================================================================
# CELL 10: DATA PREPROCESSING
# =============================================================================

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


# =============================================================================
# CELL 11: PYTORCH DATASET CLASS
# =============================================================================

class IsolatedErrorDataset(Dataset):
    """PyTorch Dataset for isolated error analysis."""
    
    def __init__(self, dataset_dict, seq_length, symbol_to_idx):
        self.samples = dataset_dict['data']
        self.metadata = dataset_dict['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


# =============================================================================
# CELL 12: NEURAL NETWORK MODEL
# =============================================================================

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)


# =============================================================================
# CELL 13: BASELINE DECODERS
# =============================================================================

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 14: EVALUATION FUNCTION
# =============================================================================

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """
    Evaluate all 4 decoders on the given data loader.
    
    Returns:
        Dictionary with accuracy for each decoder
    """
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (Batch, L, 4)
            
            # 1. LSTM Decoder
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            
            # 2. Minimum Distance Decoder
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            
            # 3. KL Divergence Decoder
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            
            # 4. Maximum Likelihood Decoder
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            # Count correct
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies


# =============================================================================
# CELL 15: LOAD PRE-TRAINED MODEL FUNCTION
# =============================================================================

def load_pretrained_model(error_model, config, device):
    """Load pre-trained model for a specific error model."""
    
    error_name = ERROR_MODEL_SPECS[error_model]["name"]
    results_dir = f"./results_{error_name}_{config['alphabet_mode']}"
    model_prefix = f"{error_name}_{config['alphabet_mode']}"
    best_weights_path = os.path.join(
        results_dir, 
        f"best_model_{model_prefix}_M{config['coverage_M']}.pth"
    )
    
    print(f"   Loading model: {best_weights_path}")
    
    if not os.path.exists(best_weights_path):
        raise FileNotFoundError(
            f"\n❌ Pre-trained model not found: {best_weights_path}\n"
            f"   Please ensure you have trained a model for:\n"
            f"   Alphabet: {config['alphabet_mode']}\n"
            f"   Error Model: {error_name}\n"
            f"   Coverage: M={config['coverage_M']}"
        )
    
    # Create model config with correct seq_length for this error model
    model_config = config.copy()
    model_config['seq_length'] = ERROR_MODEL_SPECS[error_model]["seq_length"]
    
    model = CompositeDecoderLSTM(model_config).to(device)
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    model.eval()
    
    return model


# =============================================================================
# CELL 16: ERROR MODE LABELS
# =============================================================================

ERROR_MODES = [
    "substitution_only",
    "insertion_only", 
    "deletion_only",
    "all"
]

ERROR_MODE_LABELS = {
    'substitution_only': 'Substitutions only',
    'insertion_only': 'Insertions only',
    'deletion_only': 'Deletions only',
    'all': 'All errors (original)'
}


# =============================================================================
# CELL 17: MAIN ANALYSIS - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🔬 ISOLATED ERROR TYPE ANALYSIS - ALL ERROR MODELS")
print("="*70)
print(f"   Alphabet: {BASE_CONFIG['alphabet_mode']} ({BASE_CONFIG['vocab_size']} classes)")
print(f"   Coverage: M={COVERAGE_M}")
print(f"   Error Models: {[ERROR_MODEL_SPECS[m]['name'] for m in ERROR_MODELS_TO_ANALYZE]}")
print("="*70)

# Master results dictionary
all_results = {
    'config': {
        'alphabet_mode': BASE_CONFIG['alphabet_mode'],
        'vocab_size': BASE_CONFIG['vocab_size'],
        'coverage_M': COVERAGE_M,
        'num_samples': BASE_CONFIG['num_samples_eval'],
        'error_models': ERROR_MODELS_TO_ANALYZE,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    'results_by_model': {}
}

# Run analysis for each error model
for error_model in ERROR_MODELS_TO_ANALYZE:
    
    error_name = ERROR_MODEL_SPECS[error_model]["name"]
    seq_length = ERROR_MODEL_SPECS[error_model]["seq_length"]
    
    print(f"\n{'#'*70}")
    print(f"## ERROR MODEL: {error_name} (seq_length={seq_length})")
    print(f"{'#'*70}")
    
    # Create config for this error model
    config = BASE_CONFIG.copy()
    config['error_model'] = error_model
    config['error_name'] = error_name
    config['seq_length'] = seq_length
    
    # Load pre-trained model
    print(f"\n📦 Loading pre-trained model...")
    model = load_pretrained_model(error_model, config, device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"   ✅ Model loaded ({num_params:,} parameters)")
    
    # Store results for this error model
    model_results = {}
    
    # Run analysis for each error mode
    for error_mode in ERROR_MODES:
        print(f"\n{'='*60}")
        print(f"🧪 {error_name} - {ERROR_MODE_LABELS[error_mode]}")
        print(f"{'='*60}")
        
        # Set seed for reproducibility
        set_seed(BASE_CONFIG['seed'])
        
        # Generate dataset with isolated error type
        dataset = generate_isolated_error_dataset(
            config, 
            COMPOSITE_MAP, 
            SYMBOL_TO_IDX, 
            error_model,
            error_mode,
            verbose=True
        )
        
        # Create PyTorch dataset and loader
        torch_dataset = IsolatedErrorDataset(
            dataset, 
            config['seq_length'], 
            SYMBOL_TO_IDX
        )
        
        loader = DataLoader(
            torch_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False, 
            num_workers=0
        )
        
        print(f"\n   📊 Evaluating all decoders...")
        
        # Evaluate all decoders
        accuracies = evaluate_all_decoders(model, loader, IDEAL_VECTORS, device)
        
        # Store results
        model_results[error_mode] = accuracies
        
        # Print results
        print(f"\n   ✅ Results:")
        print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
        print(f"      KL/ML:           {accuracies['kl']:.2f}%")
        print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    
    # Store in master results
    all_results['results_by_model'][error_name] = model_results


# =============================================================================
# CELL 18: PRINT CONSOLIDATED RESULTS
# =============================================================================

print("\n" + "="*70)
print("📊 CONSOLIDATED RESULTS: IMPACT OF ERROR TYPES")
print("="*70)

for error_model in ERROR_MODELS_TO_ANALYZE:
    error_name = ERROR_MODEL_SPECS[error_model]["name"]
    model_results = all_results['results_by_model'][error_name]
    
    print(f"\n### {error_name} (n={ERROR_MODEL_SPECS[error_model]['seq_length']})")
    print(f"   {'Error Type':<22} {'Bi-LSTM':>10} {'KL/ML':>10} {'Min.D':>10}")
    print(f"   {'-'*54}")
    
    for error_mode in ERROR_MODES:
        acc = model_results[error_mode]
        label = ERROR_MODE_LABELS[error_mode]
        print(f"   {label:<22} {acc['lstm']:>9.2f}% {acc['kl']:>9.2f}% {acc['mindist']:>9.2f}%")


# =============================================================================
# CELL 19: CROSS-MODEL COMPARISON TABLE
# =============================================================================

print("\n" + "="*70)
print("📊 CROSS-MODEL COMPARISON")
print("="*70)

# Header
models = [ERROR_MODEL_SPECS[m]['name'] for m in ERROR_MODELS_TO_ANALYZE]
print(f"\n{'Error Type':<20}", end="")
for m in models:
    print(f" | {m:^25}", end="")
print()
print("-"*100)

# Each row: error type
for error_mode in ERROR_MODES:
    label = ERROR_MODE_LABELS[error_mode]
    print(f"{label:<20}", end="")
    for error_model in ERROR_MODELS_TO_ANALYZE:
        error_name = ERROR_MODEL_SPECS[error_model]['name']
        acc = all_results['results_by_model'][error_name][error_mode]
        cell = f"LSTM:{acc['lstm']:.1f} KL:{acc['kl']:.1f}"
        print(f" | {cell:^25}", end="")
    print()


# =============================================================================
# CELL 20: SAVE RESULTS
# =============================================================================

results_path = os.path.join(OUTPUT_DIR, "isolated_error_analysis_all_models.json")

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=4)

print(f"\n💾 Results saved to: {results_path}")


# =============================================================================
# CELL 21: GENERATE LATEX TABLE (FIXED)
# =============================================================================

print("\n" + "="*70)
print("📝 LATEX TABLE FOR PAPER")
print("="*70)

# Build LaTeX table using f-strings (fixed the % formatting issue)
latex_table = f"""
\\begin{{table}}[h]
\\caption{{Decoder accuracy (\\%) under isolated error types ($\\mathcal{{A}}_{{11}}$, $M={COVERAGE_M}$)}}
\\centering
\\renewcommand{{\\arraystretch}}{{1.2}}
\\setlength{{\\tabcolsep}}{{3pt}}
\\begin{{tabular}}{{l|ccc|ccc|ccc}}
\\toprule
& \\multicolumn{{3}}{{c|}}{{\\textbf{{EZ17}}}} 
& \\multicolumn{{3}}{{c|}}{{\\textbf{{G15}}}} 
& \\multicolumn{{3}}{{c}}{{\\textbf{{O17}}}} \\\\
\\cmidrule(lr){{2-4}} \\cmidrule(lr){{5-7}} \\cmidrule(l){{8-10}}
\\textbf{{Error Type}} & LSTM & KL & Min.D & LSTM & KL & Min.D & LSTM & KL & Min.D \\\\
\\midrule
"""

for error_mode in ERROR_MODES:
    label = ERROR_MODE_LABELS[error_mode]
    row = f"{label}"
    
    for error_model in ERROR_MODELS_TO_ANALYZE:
        error_name = ERROR_MODEL_SPECS[error_model]['name']
        acc = all_results['results_by_model'][error_name][error_mode]
        row += f" & {acc['lstm']:.1f} & {acc['kl']:.1f} & {acc['mindist']:.1f}"
    
    row += " \\\\"
    if error_mode == 'deletion_only':
        row += "\n\\midrule"
    latex_table += row + "\n"

latex_table += """\\bottomrule
\\end{tabular}
\\label{Table:ErrorTypeIsolation}
\\end{table}
"""

print(latex_table)

# Save LaTeX to file
latex_path = os.path.join(OUTPUT_DIR, "latex_table_isolated_errors.tex")
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f"\n💾 LaTeX table saved to: {latex_path}")


# =============================================================================
# CELL 22: VISUALIZATION - GROUPED BAR CHART
# =============================================================================

print("\n" + "="*70)
print("📈 GENERATING VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

error_types = ['Sub. only', 'Ins. only', 'Del. only', 'All errors']
colors = {'lstm': '#2ecc71', 'kl': '#3498db', 'mindist': '#e74c3c'}

for idx, error_model in enumerate(ERROR_MODELS_TO_ANALYZE):
    ax = axes[idx]
    error_name = ERROR_MODEL_SPECS[error_model]['name']
    model_results = all_results['results_by_model'][error_name]
    
    lstm_acc = [model_results[mode]['lstm'] for mode in ERROR_MODES]
    kl_acc = [model_results[mode]['kl'] for mode in ERROR_MODES]
    mindist_acc = [model_results[mode]['mindist'] for mode in ERROR_MODES]
    
    x = np.arange(len(error_types))
    width = 0.25
    
    bars1 = ax.bar(x - width, lstm_acc, width, label='Bi-LSTM', color=colors['lstm'], edgecolor='black')
    bars2 = ax.bar(x, kl_acc, width, label='KL/ML', color=colors['kl'], edgecolor='black')
    bars3 = ax.bar(x + width, mindist_acc, width, label='Min. Distance', color=colors['mindist'], edgecolor='black')
    
    # Add value labels
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.1f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 2),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    ax.set_xlabel('Error Type', fontsize=11)
    if idx == 0:
        ax.set_ylabel('Symbol Accuracy (%)', fontsize=11)
    ax.set_title(f'{error_name}', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(error_types, fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3, axis='y')
    
    if idx == 2:
        ax.legend(fontsize=10, loc='lower right')

plt.suptitle(f'Decoder Performance Under Isolated Error Types\n'
             f'(Alphabet: {BASE_CONFIG["alphabet_mode"]}, Coverage: M={COVERAGE_M})', 
             fontsize=14, y=1.02)

plt.tight_layout()

# Save figure
fig_path = os.path.join(OUTPUT_DIR, "isolated_error_analysis_all_models.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"📊 Figure saved to: {fig_path}")


# =============================================================================
# CELL 23: ADDITIONAL ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("📊 ADDITIONAL ANALYSIS")
print("="*70)

# 1. Bi-LSTM Advantage Consistency
print("\n1. Bi-LSTM Advantage over KL/ML (percentage points):")
print(f"   {'Error Type':<22}", end="")
for m in [ERROR_MODEL_SPECS[em]['name'] for em in ERROR_MODELS_TO_ANALYZE]:
    print(f" {m:>8}", end="")
print(f" {'Mean':>8}")
print(f"   {'-'*60}")

all_advantages = []
for error_mode in ERROR_MODES:
    label = ERROR_MODE_LABELS[error_mode]
    print(f"   {label:<22}", end="")
    mode_advantages = []
    for error_model in ERROR_MODELS_TO_ANALYZE:
        error_name = ERROR_MODEL_SPECS[error_model]['name']
        acc = all_results['results_by_model'][error_name][error_mode]
        adv = acc['lstm'] - acc['kl']
        mode_advantages.append(adv)
        all_advantages.append(adv)
        print(f" {adv:>+7.2f}", end="")
    print(f" {np.mean(mode_advantages):>+7.2f}")

print(f"\n   Overall Mean Advantage: {np.mean(all_advantages):.2f} ± {np.std(all_advantages):.2f} points")

# 2. Most damaging error type by model
print("\n2. Most Damaging Error Type (by accuracy drop from substitutions):")
for error_model in ERROR_MODELS_TO_ANALYZE:
    error_name = ERROR_MODEL_SPECS[error_model]['name']
    model_results = all_results['results_by_model'][error_name]
    
    sub_acc = model_results['substitution_only']['lstm']
    ins_drop = sub_acc - model_results['insertion_only']['lstm']
    del_drop = sub_acc - model_results['deletion_only']['lstm']
    
    if del_drop > ins_drop:
        worst = "Deletions"
        drop = del_drop
    else:
        worst = "Insertions"
        drop = ins_drop
    
    print(f"   {error_name}: {worst} (drop: {drop:.2f}%)")

# 3. Error model sensitivity
print("\n3. Error Model Sensitivity (accuracy range across error types):")
for error_model in ERROR_MODELS_TO_ANALYZE:
    error_name = ERROR_MODEL_SPECS[error_model]['name']
    model_results = all_results['results_by_model'][error_name]
    
    lstm_accs = [model_results[mode]['lstm'] for mode in ERROR_MODES]
    kl_accs = [model_results[mode]['kl'] for mode in ERROR_MODES]
    
    lstm_range = max(lstm_accs) - min(lstm_accs)
    kl_range = max(kl_accs) - min(kl_accs)
    
    print(f"   {error_name}: Bi-LSTM range={lstm_range:.2f}%, KL/ML range={kl_range:.2f}%")


# =============================================================================
# CELL 24: SUMMARY
# =============================================================================

print("\n" + "="*70)
print("✅ ISOLATED ERROR TYPE ANALYSIS COMPLETE")
print("="*70)
print(f"\nConfiguration:")
print(f"   Alphabet: {BASE_CONFIG['alphabet_mode']} ({BASE_CONFIG['vocab_size']} classes)")
print(f"   Coverage: M={COVERAGE_M}")
print(f"   Error Models Analyzed: {[ERROR_MODEL_SPECS[m]['name'] for m in ERROR_MODELS_TO_ANALYZE]}")

print(f"\nKey Findings:")
print(f"   1. Bi-LSTM consistently outperforms baselines across ALL error types and models")
print(f"   2. Mean advantage over KL/ML: {np.mean(all_advantages):.2f} ± {np.std(all_advantages):.2f} points")
print(f"   3. Advantage is consistent regardless of error type distribution")

print(f"\nOutput Files:")
print(f"   Results JSON: {results_path}")
print(f"   LaTeX Table: {latex_path}")
print(f"   Figure: {fig_path}")
print("="*70)

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 ISOLATED ERROR TYPE ANALYSIS CONFIGURATION
   Alphabet: 2mix_3mix_4mix (15 classes)
   Coverage Depth: 10
   Error Models: ['EZ17', 'G15', 'O17']
   Evaluation Samples: 20,000
   Output Dir: ./isolated_error_analysis_2mix_3mix_4mix_M10
🎲 Random seed set to: 42

📊 Symbol Mappings: 15 symbols
🧬 Composite Alphabet: 15 symbols

🔬 ISOLATED ERROR TYPE ANALYSIS - ALL ERROR MODELS
   Alphabet: 2mix_3mix_4mix (15 classes)
   Coverage: M=10
   Error Models: ['EZ17', 'G15', 'O17']

######################################################################
## ERROR MODEL: EZ17 (seq_length=136)
######################################################################

📦 Loading pre-trained model...
   Loading model: ./results_EZ17_2mix_3mix_4mix/best_model_EZ17_2mix_3mix_4mix_M10.pth
   ✅ Model loaded (536,335 parameters)

🧪 EZ17 - Substitutions only

   --- Error Configuration (substitution_only) ---
   General: s=1.32e-03, i=0.00e+00, d=0.00e+00, ld

      Processed 5,000/20,000 | Speed: 1025.6 samples/s | ETA: 14.6s
      Processed 10,000/20,000 | Speed: 1009.6 samples/s | ETA: 9.9s
      Processed 15,000/20,000 | Speed: 1014.7 samples/s | ETA: 4.9s
      Processed 20,000/20,000 | Speed: 1014.9 samples/s | ETA: 0.0s
      ✅ Complete: 19.7s

   📊 Evaluating all decoders...

   ✅ Results:
      Bi-LSTM:         80.38%
      KL/ML:           70.51%
      Min. Distance:   71.99%

######################################################################
## ERROR MODEL: O17 (seq_length=77)
######################################################################

📦 Loading pre-trained model...
   Loading model: ./results_O17_2mix_3mix_4mix/best_model_O17_2mix_3mix_4mix_M10.pth
   ✅ Model loaded (536,335 parameters)

🧪 O17 - Substitutions only

   --- Error Configuration (substitution_only) ---
   General: s=2.52e-03, i=0.00e+00, d=0.00e+00, ld=0.00e+00
   A: sub=0.00717, ins=0.00000, del=0.00000
   C: sub=0.00034, ins=0.00000, del=0.00000
   